In [18]:
import os
import sys
import time

In [19]:
import importlib
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import library.uart3_protocol as uart3_protocol
import library.aes_gcm_app as aes_gcm_app
importlib.reload(uart3_protocol)
importlib.reload(aes_gcm_app)

from library.uart3_protocol import UART3Protocol
from library.aes_gcm_app import STM32AESGCM

print("Library updated / reloaded")

Library updated / reloaded


In [20]:
PORT = "/dev/cu.usbmodem1103"
BAUD = 1000000
MAX_BUFFER_SIZE = 32768
TIMEOUT = 0.01

uart = UART3Protocol(
    port = PORT,
    baud = BAUD,
    max_buffer_size = MAX_BUFFER_SIZE,
    timeout = TIMEOUT
)

uart.open()
time.sleep(1)

In [21]:
key = bytes(range(32))
aes = STM32AESGCM(uart, key)

In [22]:
# 清 Python 端 buffer
aes.clear()
print("CLEAR STM32 UART BUFFER")

CLEAR STM32 UART BUFFER


In [23]:
def run_aes_gcm_stress_test(
    aes,
    uart,
    test_method,
    total=50,
    size=None,
    sleep_time=0.01,
    auto_resync=True,
):
    """
    test_method examples:
        aes.stm32_encrypt_stm32_decrypt_test
        aes.stm32_encrypt_local_decrypt_test
        aes.local_encrypt_stm32_decrypt_test
    """

    if size is None:
        size = uart.max_buffer_size

    ok_count = 0
    error_count = 0
    last_elapsed = 0.0

    start_all = time.perf_counter()

    method_name = getattr(test_method, "__name__", str(test_method))
    print(f"Method: {method_name}")
    print(f"Total : {total}")
    print(f"Size  : {size} bytes")
    print()

    for i in range(total):
        data = os.urandom(size)

        try:
            result = test_method(data)

            last_elapsed = result["elapsed"]

            if result["ok"]:
                ok_count += 1
            else:
                error_count += 1
                print()
                print(f"ERROR at {i + 1}/{total}")
                print("nonce:", result["nonce"].hex())
                print("tag:", result["tag"].hex())
                print("single elapsed:", result["elapsed"])

        except Exception as e:
            error_count += 1
            print()
            print(f"EXCEPTION at {i + 1}/{total}: {e}")

            if auto_resync:
                try:
                    uart.clear_buffers()
                    uart.write_line("CLEAR")
                    aes._wait_ready(timeout=2.0)
                    print("resync OK")
                except Exception as clear_error:
                    print("resync failed:", clear_error)

        done = i + 1
        percent = done / total * 100

        total_elapsed = time.perf_counter() - start_all
        avg_time = total_elapsed / done
        eta = avg_time * (total - done)

        print(
            f"Progress: {done}/{total} "
            f"({percent:6.2f}%) | "
            f"OK={ok_count} | ERR={error_count} | "
            f"last={last_elapsed:7.3f}s | "
            f"total={total_elapsed:8.2f}s | "
            f"avg={avg_time:7.3f}s | "
            f"ETA={eta:8.2f}s", # Estimated Time Arrival(預估剩餘時間)
            end="\r",
            flush=True
        )

        time.sleep(sleep_time)

    total_elapsed = time.perf_counter() - start_all

    print()
    print(f"Total time: {total_elapsed:.2f} s")
    print(f"Average time per test: {total_elapsed / total:.4f} s")

    if error_count == 0:
        print("NO ERROR")
    else:
        print(f"HAS ERROR: {error_count}")

    return {
        "method": method_name,
        "total": total,
        "size": size,
        "ok_count": ok_count,
        "error_count": error_count,
        "total_elapsed": total_elapsed,
        "avg_time": total_elapsed / total,
    }

print("build funtion: run_aes_gcm_stress_test")

build funtion: run_aes_gcm_stress_test


In [24]:
methods = [
    aes.stm32_encrypt_stm32_decrypt_test,
    aes.stm32_encrypt_local_decrypt_test,
    aes.local_encrypt_stm32_decrypt_test,
]

for method in methods:
    print("-" * 150)
    result = run_aes_gcm_stress_test(
        aes=aes,
        uart=uart,
        test_method=method,
        total=1,
        size=uart.max_buffer_size,
    )

    print(result)

------------------------------------------------------------------------------------------------------------------------------------------------------
Method: stm32_encrypt_stm32_decrypt_test
Total : 1
Size  : 32768 bytes

Progress: 1/1 (100.00%) | OK=1 | ERR=0 | last=  2.218s | total=    2.22s | avg=  2.219s | ETA=    0.00s
Total time: 2.23 s
Average time per test: 2.2319 s
NO ERROR
{'method': 'stm32_encrypt_stm32_decrypt_test', 'total': 1, 'size': 32768, 'ok_count': 1, 'error_count': 0, 'total_elapsed': 2.231892833020538, 'avg_time': 2.231892833020538}
------------------------------------------------------------------------------------------------------------------------------------------------------
Method: stm32_encrypt_local_decrypt_test
Total : 1
Size  : 32768 bytes

Progress: 1/1 (100.00%) | OK=1 | ERR=0 | last=  1.107s | total=    1.11s | avg=  1.108s | ETA=    0.00s
Total time: 1.12 s
Average time per test: 1.1188 s
NO ERROR
{'method': 'stm32_encrypt_local_decrypt_test', 'tota

In [25]:
uart.close()